In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import brier_score_loss, roc_auc_score
from help.ckm_utils import CONDITIONS
from help.stats_utils import bootstrap, calibration_table

OUT_DIR, COND = "Results_MoE", list(CONDITIONS)
oof = pd.read_csv(f"{OUT_DIR}/oof_predictions.csv")
assert oof["participant_id"].is_unique

In [ ]:
# ---- 1. Discrimination, calibration, sensitivity/specificity (95% bootstrap CIs) ----
def entities(d):
    yield ("MoE", d["y_any"].to_numpy(), d["predicted_composite_score"].to_numpy() / 100,
           d["flag_rate_composite"].to_numpy())
    for c in COND:
        pool = (d[f"true_{c}"] == 1) | (d["burden_score"] == 0)
        yield (f"Expert: {c}", d.loc[pool, f"true_{c}"].to_numpy(), d.loc[pool, f"prob_{c}"].to_numpy(),
               d.loc[pool, f"flag_rate_{c}"].to_numpy())

def auc_brier(y, s):
    return {
        "AUC": roc_auc_score(y, s),
        "Brier": brier_score_loss(y, s),
    }

def sens_spec(y, flag_rate):
    """Thresholds were fixed inside each outer fold; mean flag rate = mean over repeats of sens / (1 - spec)."""
    return {"Sensitivity": flag_rate[y == 1].mean(), "Specificity": 1 - flag_rate[y == 0].mean()}

metrics, curves = [], []
for sex, d in oof.groupby("gender"):
    for model, y, s, fl in entities(d):
        y = y.astype(int)
        meta = dict(gender=sex, model=model, n=len(y), n_cases=int(y.sum()))
        metrics += [bootstrap(auc_brier, y, s).assign(**meta), bootstrap(sens_spec, y, fl).assign(**meta)]
        curves.append(calibration_table(y, s).assign(gender=sex, model=model))
metrics_df = pd.concat(metrics, ignore_index=True)
metrics_df.to_csv(f"{OUT_DIR}/validation_metrics.csv", index=False)
pd.concat(curves).to_csv(f"{OUT_DIR}/calibration_curves.csv", index=False)

keep = ["AUC", "Brier", "Sensitivity", "Specificity"]
print("MoE risk score, headline (repeat-averaged OOF score):")
print(metrics_df.query("model == 'MoE' and metric in @keep")
      [["gender", "metric", "estimate", "ci_lo", "ci_hi"]].round(3).to_string(index=False))

print("\nExperts, headline (repeat-averaged OOF score):")
print(metrics_df.query("model != 'MoE' and metric in @keep")
      [["gender", "model", "metric", "estimate", "ci_lo", "ci_hi"]].round(3).to_string(index=False))

stab = pd.read_csv(f"{OUT_DIR}/cv_stability.csv").groupby("gender")["auc"].agg(["mean", "std"]).round(4)
print("\nCV stability (mean/SD of 10 single-repeat AUCs):\n", stab)

In [ ]:
# ---- 1b. Attribution accuracy: isolated cases (top-1) and multimorbid recall@k ----
def bootstrap_ci(values, n_boot=1000, ci=0.95, seed=42):
    """Bootstrap CI for a mean of a binary/continuous array."""
    values = np.asarray(values)
    values = values[~np.isnan(values)]
    if len(values) == 0:
        return (np.nan, np.nan)
    rng = np.random.default_rng(seed)
    boot_means = np.array([
        rng.choice(values, size=len(values), replace=True).mean()
        for _ in range(n_boot)
    ])
    alpha = (1 - ci) / 2
    lo, hi = np.quantile(boot_means, [alpha, 1 - alpha])
    return (lo, hi)

attr_rows = []
for sex, d in oof.groupby("gender"):
    iso = d[d["is_isolated_case"]]
    multi = d[d["is_multimorbid"]]

    top1_acc = iso["attribution_correct"].mean()
    top1_lo, top1_hi = bootstrap_ci(iso["attribution_correct"].values)

    recall_mean = multi["recall_at_k_score"].mean()
    recall_lo, recall_hi = bootstrap_ci(multi["recall_at_k_score"].values)

    attr_rows.append({
        "gender": sex,
        "n_isolated": len(iso),
        "top1_attribution_accuracy": top1_acc,
        "top1_ci_lo": top1_lo,
        "top1_ci_hi": top1_hi,
        "n_multimorbid": len(multi),
        "mean_recall_at_k": recall_mean,
        "recall_ci_lo": recall_lo,
        "recall_ci_hi": recall_hi,
    })

attribution_summary = pd.DataFrame(attr_rows)
attribution_summary.to_csv(f"{OUT_DIR}/attribution_summary.csv", index=False)
print("\nAttribution accuracy (isolated cases: top-1 correct; multimorbid: recall@k) with 95% bootstrap CIs:")
print(attribution_summary.round(3).to_string(index=False))

In [ ]:
# ---- 1c. Balanced accuracy (Youden threshold, fixed in outer folds) ----
def balanced_accuracy(y, flag_rate):
    """(Sensitivity + Specificity) / 2 -- robust to class imbalance, unlike raw accuracy."""
    sens = flag_rate[y == 1].mean()
    spec = 1 - flag_rate[y == 0].mean()
    return {"Balanced accuracy": 0.5 * (sens + spec)}

ba_rows = []
for sex, d in oof.groupby("gender"):
    for model, y, s, fl in entities(d):
        y = y.astype(int)
        meta = dict(gender=sex, model=model, n=len(y), n_cases=int(y.sum()))
        ba_rows.append(bootstrap(balanced_accuracy, y, fl).assign(**meta))

ba_df = pd.concat(ba_rows, ignore_index=True)
ba_df.to_csv(f"{OUT_DIR}/balanced_accuracy.csv", index=False)

print("\nBalanced accuracy (Youden threshold, fixed in outer folds; 95% bootstrap CIs):")
print("MetaVoX:")
print(ba_df.query("model == 'MoE'")
      [["gender", "metric", "estimate", "ci_lo", "ci_hi"]].round(3).to_string(index=False))
print("\nExperts:")
print(ba_df.query("model != 'MoE'")
      [["gender", "model", "metric", "estimate", "ci_lo", "ci_hi"]].round(3).to_string(index=False))

In [ ]:
# ---- 2. Calibration-in-the-large and gradient across burden levels ----
rows = []
for sex, d in oof.groupby("gender"):
    p, y, b = d["predicted_composite_score"].to_numpy() / 100, d["y_any"].to_numpy(), d["burden_score"].to_numpy()
    rows.append(bootstrap(lambda p_, y_: {"Mean predicted": p_.mean(), "Observed prevalence": y_.mean(),
                                          "CITL (pred - obs)": p_.mean() - y_.mean()},
                          p, y).assign(gender=sex, group="all", n=len(p)))
    for lvl in sorted(np.unique(b)):
        m = b == lvl
        rows.append(bootstrap(lambda p_: {"Mean predicted": p_.mean()}, p[m])
                    .assign(gender=sex, group=f"burden {lvl}", n=int(m.sum())))
grad = pd.concat(rows, ignore_index=True)
grad.to_csv(f"{OUT_DIR}/composite_calibration_and_gradient.csv", index=False)
print("\nMean predicted probability by burden level:")
print(grad[grad.metric == "Mean predicted"].pivot(index="group", columns="gender", values="estimate").round(3))

In [ ]:
# ---- 3. Per-condition prevalence by burden score, by gender ----
rows = []
for (sex, burden), g in oof.groupby(["gender", "burden_score"]):
    if burden in (0, 4):     # 0: no conditions by definition; 4: trivially 100% for all -- both uninformative
        continue
    for c in COND:
        rows.append({"gender": sex, "burden_score": int(burden), "condition": c,
                     "pct_with_condition": 100 * (g[f"true_{c}"] == 1).mean(), "n": len(g)})
prevalence_by_burden = pd.DataFrame(rows)
prevalence_by_burden.to_csv(f"{OUT_DIR}/condition_prevalence_by_burden.csv", index=False)
print("\nPer-condition prevalence by burden score (Table S_X):")
print(prevalence_by_burden.pivot_table(index=["gender", "burden_score"], columns="condition",
                                       values="pct_with_condition").round(1))

In [ ]:
# ---- Distribution of isolated conditions ----
rows = []
for sex, d in oof.groupby("gender"):
    iso = d[d["is_isolated_case"]]
    n_iso = len(iso)
    for c in COND:
        n = int((iso[f"true_{c}"] == 1).sum())
        rows.append({
            "gender": sex,
            "condition": c,
            "n_isolated": n,
            "pct_isolated": 100 * n / n_iso if n_iso else np.nan,
            "total_isolated_cases": n_iso, })

isolated_dist = pd.DataFrame(rows)
isolated_dist.to_csv(f"{OUT_DIR}/isolated_condition_distribution.csv",index=False)

print("\nDistribution of isolated conditions:")
print(isolated_dist.pivot(
        index="condition",
        columns="gender",
        values="pct_isolated").round(1))

In [ ]:
# %%
# ---- 4. Expert x condition AUC matrix ----
def cross_auc_matrix(P, pos, ctrl):
    """A[j, k] = AUC of expert j's probability discriminating condition k vs burden-0 controls."""
    A = np.zeros((len(COND), len(COND)))
    for k in range(len(COND)):
        m = pos[:, k] | ctrl
        for j in range(len(COND)):
            A[j, k] = roc_auc_score(pos[m, k].astype(int), P[m, j])
    return A

matrices = []
for sex, d in oof.groupby("gender"):
    P = d[[f"prob_{c}" for c in COND]].to_numpy()
    pos = d[[f"true_{c}" for c in COND]].to_numpy().astype(bool)
    ctrl = (d["burden_score"] == 0).to_numpy()

    A = cross_auc_matrix(P, pos, ctrl)
    matrices.append(pd.DataFrame(A, index=COND, columns=COND).assign(gender=sex, expert=COND).set_index(["gender", "expert"]))

pd.concat(matrices).round(3).to_csv(f"{OUT_DIR}/expert_condition_auc_matrix.csv")

print("\nFull expert x condition AUC matrix (rows = expert probability, columns = condition):")
for sex, m in pd.concat(matrices).groupby("gender"):
    print(f"\n{sex}:")
    print(m.droplevel("gender").round(3))

# ---- 4b. Own-condition vs other-conditions AUC gap (supports "nearly as well as own target" / obesity text) ----
gap_rows = []
for sex, d in oof.groupby("gender"):
    P = d[[f"prob_{c}" for c in COND]].to_numpy()
    pos = d[[f"true_{c}" for c in COND]].to_numpy().astype(bool)
    ctrl = (d["burden_score"] == 0).to_numpy()
    A = cross_auc_matrix(P, pos, ctrl)
    for j, c in enumerate(COND):
        gap_rows.append({"gender": sex, "expert": c,
                         "own_auc": A[j, j],
                         "mean_auc_other_conditions": np.delete(A[j, :], j).mean()})
gap_df = pd.DataFrame(gap_rows)
gap_df.to_csv(f"{OUT_DIR}/expert_condition_auc_gap.csv", index=False)
print("\nOwn-condition AUC vs mean AUC on the other 3 conditions:")
print(gap_df.round(3).to_string(index=False))

In [ ]:
# ---- Bootstrap comparison of obesity detection: obesity expert vs other experts ----
from help.stats_utils import SEED, N_BOOT

rows = []
for sex, d in oof.groupby("gender"):
    ctrl = (d["burden_score"] == 0).to_numpy()
    obesity = (d["true_BMI>=30"] == 1).to_numpy()

    mask = obesity | ctrl
    y = obesity[mask].astype(int)

    obesity_scores = d.loc[mask, "prob_BMI>=30"].to_numpy()
    obesity_auc = roc_auc_score(y, obesity_scores)

    rng = np.random.default_rng(SEED)

    for other in [c for c in COND if c != "BMI>=30"]:
        other_scores = d.loc[mask, f"prob_{other}"].to_numpy()
        other_auc = roc_auc_score(y, other_scores)
        delta = obesity_auc - other_auc

        boot_delta = []
        n = len(y)

        for _ in range(N_BOOT):
            idx = rng.integers(0, n, n)
            try:
                auc_ob = roc_auc_score(y[idx], obesity_scores[idx])
                auc_ot = roc_auc_score(y[idx], other_scores[idx])
                boot_delta.append(auc_ob - auc_ot)
            except ValueError:
                continue

        ci_lo, ci_hi = np.percentile(boot_delta, [2.5, 97.5])

        p_boot = max(
            2 * min(
                (np.array(boot_delta) <= 0).mean(),
                (np.array(boot_delta) >= 0).mean()),
            1 / (len(boot_delta) + 1))

        rows.append({"gender": sex,
            "target_condition": "obesity",
            "reference_expert": "obesity",
            "comparison_expert": other,
            "obesity_expert_auc": obesity_auc,
            "other_expert_auc": other_auc,
            "delta_auc": delta,
            "ci_lo": ci_lo,
            "ci_hi": ci_hi,
            "p_boot": p_boot,
            "ci_excludes_0": (ci_lo > 0) | (ci_hi < 0) })

obesity_auc_comparisons = pd.DataFrame(rows)
obesity_auc_comparisons.to_csv(f"{OUT_DIR}/obesity_expert_auc_comparisons.csv", index=False)

print("\nObesity expert vs other experts for detecting obesity:")
print(obesity_auc_comparisons[[
            "gender",
            "comparison_expert",
            "delta_auc",
            "ci_lo",
            "ci_hi",
            "p_boot"]].round(4).to_string(index=False))

In [ ]:
# ---- 5. Spearman of each expert score with age, among burden-0 controls ----
from scipy.stats import spearmanr

pop = pd.read_pickle("analytic_population.pkl")
d_all = oof.merge(pop[["participant_id", "age"]], on="participant_id", how="left", validate="1:1")
assert d_all["age"].notna().all()

rho_rows = []
for sex, d in d_all.groupby("gender"):
    ctrl = d[d.burden_score == 0]
    for c in COND:
        rho, _ = spearmanr(ctrl["age"], ctrl[f"prob_{c}"])
        rho_rows.append({"gender": sex, "expert": c, "spearman_with_age_in_controls": rho})
age_corr = pd.DataFrame(rho_rows)
age_corr.to_csv(f"{OUT_DIR}/expert_age_correlation_in_controls.csv", index=False)
print("\nSpearman of each expert score with age among burden-0 participants (far from 0 => expert encodes age):")
print(age_corr.round(2).to_string(index=False))

In [ ]:
# ---- 6. Spearman: additional metabolic conditions vs expert predicted probability ----
from statsmodels.stats.multitest import multipletests

spearman_rows = []
for sex, d in oof.groupby("gender"):
    for c in COND:
        has_c = d[f"true_{c}"] == 1
        extra_burden = d.loc[has_c, "burden_score"] - 1
        prob = d.loc[has_c, f"prob_{c}"]
        rho_c, p_c = spearmanr(extra_burden, prob)
        spearman_rows.append({"gender": sex, "condition": c, "outcome": "continuous prob", "rho": rho_c, "p_value": p_c, "n": has_c.sum()})

spearman_df = pd.DataFrame(spearman_rows)
spearman_df["q_fdr"] = multipletests(
    spearman_df["p_value"],
    method="fdr_bh")[1]

spearman_df.to_csv(f"{OUT_DIR}/burden_vs_expert_probability_spearman.csv", index=False)
print("\nSpearman: additional conditions vs expert predicted probability (FDR q per outcome type):")
print(spearman_df.round(4).to_string(index=False))

In [ ]:
# ---- Spearman p-values: burden_score vs predicted_composite_score ----
comp_check = pd.read_csv("Results_MoE/oof_predictions.csv")
comp_check["gender"] = comp_check["gender"].str.lower().str.strip().str.capitalize()

print("Exact Spearman p-values (burden_score vs predicted_composite_score):")
for gender in ["Female", "Male"]:
    d = comp_check[comp_check.gender == gender]
    rho, p = spearmanr(d.burden_score, d.predicted_composite_score)
    print(f"  {gender}: rho={rho:.4f}, p={p:.4e}")